> **Version étudiant** — les cellules d'exercice ne rappellent que la consigne : le code est à écrire entièrement par vous-même, sans squelette imposé. Un exemple travaillé sur un cas analogue précède toujours ce type d'exercice. Les cellules repérées par **Question** n'ont pas de correction automatique : exécutez le code fourni, observez, et répondez par écrit. La version corrigée est téléchargeable depuis la page du cours.

# Machines à vecteurs de support (SVM)

**Notebook 5/9 — Introduction à l'apprentissage supervisé**
*Guillaume Metzler — Université Lyon 2 (L3 MIASHS → Master)*

Ce notebook porte sur :
- le SVM à **marge dure**, pour des données parfaitement séparables,
- le SVM à **marge souple** (variables d'écart, hyperparamètre `C`) et sa
  formulation duale,
- l'**astuce du noyau** (*kernel trick* : linéaire, polynomial, gaussien/RBF)
  pour les données qui ne le sont pas.

Parmi tous les hyperplans qui séparent deux classes, le SVM choisit celui qui
**maximise la marge**, c'est-à-dire la distance entre la frontière et les
points d'entraînement les plus proches. L'idée est simple, mais elle
débouche sur un problème d'optimisation convexe, une formulation duale, et
une astuce qui permet de traiter des données non linéairement séparables
sans jamais construire explicitement un nouvel espace de représentation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.datasets import make_blobs, make_circles, make_moons, load_wine, load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

np.random.seed(0)
plt.rcParams["figure.figsize"] = (6, 5)


## 1. Marge maximale et SVM à marge dure

Un SVM linéaire classe un exemple $x$ selon le signe de son score :

$$h(x) = \text{sign}\big(\langle w, x\rangle + b\big).$$

L'ensemble $\{x : \langle w, x\rangle + b = 0\}$ est un hyperplan qui sépare
l'espace en deux régions. Quand les données sont linéairement séparables, il
en existe en général une infinité de valides : un séparateur trop proche
d'une des deux classes généralise mal, alors qu'un séparateur « au milieu »
des deux nuages de points est plus robuste à de nouvelles observations.
C'est cette intuition — choisir le séparateur le plus éloigné des deux
classes — qui définit le SVM.

En normalisant $(w,b)$, les deux hyperplans qui bordent la marge s'écrivent
$\langle w,x\rangle+b=\pm1$. On montre (voir le cours) que la distance entre
ces deux hyperplans, la marge $\gamma$, vaut :

$$\gamma = \frac{2}{\|w\|_2}.$$

Maximiser $\gamma$ revient donc à minimiser $\|w\|_2$, ou, de façon
équivalente et plus commode numériquement, $\tfrac12\|w\|_2^2$.

### SVM à marge dure

$$\min_{(w,b)\in\mathbb{R}^{d+1}} \ \frac12\|w\|_2^2
\qquad \text{s.c.} \qquad y_i(\langle w, x_i\rangle + b) \ge 1,\ \ \forall i=1,\dots,m.$$

Les points $x_i$ pour lesquels la contrainte est active
($y_i(\langle w,x_i\rangle+b)=1$, exactement sur la marge) sont les
**vecteurs de support** : ce sont eux, et eux seuls, qui déterminent
l'hyperplan. La distance d'un point $x$ quelconque à l'hyperplan
$\langle w,x\rangle+b=0$ est $\dfrac{|\langle w,x\rangle+b|}{\|w\|_2}$.

In [ ]:
def plot_svm_boundary(clf, X, y, ax, title):
    '''Trace la frontiere de decision, la marge, et les vecteurs de support.'''
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c="tab:blue", edgecolor="k",
               label="classe 0", zorder=3)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c="tab:red", edgecolor="k",
               label="classe 1", zorder=3)

    xlim, ylim = ax.get_xlim(), ax.get_ylim()
    xx = np.linspace(*xlim, 200)
    yy = np.linspace(*ylim, 200)
    YY, XX = np.meshgrid(yy, xx)
    grid = np.c_[XX.ravel(), YY.ravel()]
    Z = clf.decision_function(grid).reshape(XX.shape)

    ax.contour(XX, YY, Z, colors="k", levels=[-1, 0, 1],
               linestyles=["--", "-", "--"], linewidths=[1.2, 1.8, 1.2])
    ax.contourf(XX, YY, Z, levels=[-1e9, 0, 1e9], colors=["#dbe9ff", "#ffdbdb"],
                alpha=0.35, zorder=0)
    ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
               s=180, facecolors="none", edgecolors="black", linewidths=1.5,
               label="vecteurs de support", zorder=4)

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_title(title)
    ax.legend(loc="best", fontsize=8)


X_sep, y_sep = make_blobs(n_samples=60, centers=2, cluster_std=0.8, random_state=42)

clf_hard = SVC(kernel="linear", C=1e6)  # C tres grand : proche du comportement "marge dure"
clf_hard.fit(X_sep, y_sep)

w = clf_hard.coef_[0]
b = clf_hard.intercept_[0]
marge_gamma = 2 / np.linalg.norm(w)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
plot_svm_boundary(clf_hard, X_sep, y_sep, ax,
                   f"SVM linéaire à marge dure (marge = {marge_gamma:.3f})")
plt.tight_layout()
plt.show()

print(f"w = {w}, b = {b:.4f}")
print(f"||w||_2 = {np.linalg.norm(w):.4f}, marge gamma = {marge_gamma:.4f}")
print(f"Nombre de vecteurs de support : {len(clf_hard.support_vectors_)} (sur {len(X_sep)} points)")


Retirons maintenant un point qui n'est **pas** vecteur de support, et
regardons si l'hyperplan change.

In [ ]:
idx_non_sv = [i for i in range(len(X_sep)) if i not in clf_hard.support_][0]
X_sep_reduit = np.delete(X_sep, idx_non_sv, axis=0)
y_sep_reduit = np.delete(y_sep, idx_non_sv)

clf_hard_reduit = SVC(kernel="linear", C=1e6)
clf_hard_reduit.fit(X_sep_reduit, y_sep_reduit)

print(f"w avant suppression  = {clf_hard.coef_[0]}")
print(f"w apres suppression  = {clf_hard_reduit.coef_[0]}")
print(f"b avant suppression  = {b:.4f}")
print(f"b apres suppression  = {clf_hard_reduit.intercept_[0]:.4f}")
print(f"Nombre de vecteurs de support avant/apres : "
      f"{len(clf_hard.support_vectors_)} / {len(clf_hard_reduit.support_vectors_)}")


$$ $$

**Question 1 :** Le titre du graphique affiche une marge calculée avec la formule $\gamma = 2/\|w\|_2$. Est-elle cohérente avec l'écartement visuel des deux droites en pointillés sur la figure ?

$$ $$

$$ $$

**Question 2 :** Après suppression du point qui n'était pas vecteur de support, w et b ont-ils changé ? Que se serait-il passé, à votre avis, si on avait retiré à la place un des points entourés sur le graphique ?

$$ $$

### Exercice 1

Générez un nouveau jeu de données linéairement séparable avec `make_blobs`
(à vous de choisir `cluster_std` et `random_state` — visez deux classes
franchement séparées), entraînez un SVM linéaire à marge dure comme
ci-dessus, affichez la figure avec `plot_svm_boundary`, et calculez « à la
main » (avec `np.linalg.norm`) la norme de $w$ ainsi que la marge $\gamma$.

In [ ]:
# Générez un jeu de données séparable (make_blobs), entraînez un SVC(kernel="linear", C=1e6),
# affichez le graphique avec plot_svm_boundary, et calculez ||w||_2 et gamma = 2/||w||_2.


### Exercice 2

En reprenant le modèle entraîné à l'exercice précédent, affichez le nombre
de vecteurs de support, calculez leurs scores via `decision_function`, et
vérifiez qu'ils valent (à peu près) $\pm1$. Comparez ce nombre au nombre
total de points d'entraînement : pourquoi est-il en général très petit
devant $m$ ?

In [ ]:
# En reprenant clf_ex1, affichez le nombre de vecteurs de support et leurs scores
# (decision_function), et vérifiez que ces scores valent bien +/-1.


## 2. SVM à marge souple

En pratique, les deux classes ne sont presque jamais parfaitement
séparables. On introduit des **variables d'écart** (*slack variables*)
$\xi=(\xi_1,\dots,\xi_m)$, qui mesurent la violation, par chaque point
$x_i$, de la contrainte $y_i(\langle w,x_i\rangle+b)\ge1$ :

$$\min_{\xi\in\mathbb{R}^m,\,(w,b)} \ \frac12\|w\|_2^2 + \frac{C}{m}\sum_{i=1}^m \xi_i
\qquad \text{s.c.} \quad y_i(\langle w,x_i\rangle+b) \ge 1-\xi_i,\ \ \xi_i \ge 0.$$

L'hyperparamètre $C>0$ règle l'arbitrage entre une marge large (peu de
contrainte sur $w$, $C$ petit) et une classification stricte des points
d'entraînement, quitte à réduire la marge ($C$ grand — on retrouve le
comportement de la marge dure quand $C\to\infty$).

**Formulation équivalente.** On peut réécrire ce problème sans les $\xi_i$,
avec la **perte charnière** (*hinge loss*) $[u]_+=\max(0,u)$ :

$$\min_{(w,b)} \ \frac12\|w\|_2^2 + \frac{C}{m}\sum_{i=1}^m \big[1-y_i(\langle w,x_i\rangle+b)\big]_+.$$

En effet $\xi_i=0$ si la contrainte dure est vérifiée, et
$\xi_i=1-y_i(\langle w,x_i\rangle+b)$ sinon — exactement la définition du
terme $[\cdot]_+$.

### Formulation duale (pour mémoire)

En introduisant le Lagrangien et les conditions KKT, on obtient le problème
dual, qui ne fait intervenir les $x_i$ que par leur produit scalaire :

$$\max_{\alpha}\ -\frac12\sum_{i,j} \alpha_i\alpha_j y_i y_j \langle x_i,x_j\rangle + \sum_i \alpha_i
\qquad \text{s.c.} \quad 0\le \alpha_i \le \frac{C}{m}, \qquad \sum_i y_i\alpha_i = 0.$$

Les conditions de complémentarité classent chaque point selon son
$\alpha_i$ : $\alpha_i=0$ pour un point correctement classé hors marge,
$\alpha_i=C/m$ pour un point mal classé ou à l'intérieur de la marge, et
$0<\alpha_i<C/m$ pour un point exactement sur la marge. Seuls les points
avec $\alpha_i>0$ — les vecteurs de support — interviennent dans
$w=\sum_i\alpha_i y_i x_i$. C'est cette propriété qui donne son nom à
l'algorithme, et qui permettra, à la section suivante, de remplacer le
produit scalaire par un noyau.

On génère des données **légèrement chevauchantes** et on observe l'effet de
$C\in\{0.01,1,100\}$ sur la marge, le nombre de vecteurs de support, et le
nombre d'erreurs d'entraînement.

In [ ]:
X_soft, y_soft = make_blobs(n_samples=100, centers=2, cluster_std=2.3, random_state=7)

C_values = [0.01, 1, 100]
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharex=True, sharey=True)

for ax, C in zip(axes, C_values):
    clf = SVC(kernel="linear", C=C)
    clf.fit(X_soft, y_soft)
    marge = 2 / np.linalg.norm(clf.coef_[0])
    n_err = np.sum(clf.predict(X_soft) != y_soft)
    plot_svm_boundary(clf, X_soft, y_soft, ax,
                       f"C = {C}\nmarge = {marge:.2f}, "
                       f"{len(clf.support_vectors_)} vect. support, {n_err} erreurs")

plt.tight_layout()
plt.show()


$$ $$

**Question 3 :** Comment évoluent la marge et le nombre de vecteurs de support quand $C$ passe de $0.01$ à $100$ ? Formulez le compromis en une phrase.

$$ $$

$$ $$

**Question 4 :** À $C=100$, le nombre d'erreurs d'entraînement est-il nul ? Est-ce surprenant, sachant que les deux classes se chevauchent nettement sur la figure ?

$$ $$

On peut aussi observer la perte charnière directement, plutôt que le
compte d'erreurs 0-1. Pour $y=1$ et un score de $0.4$, ou $y=-1$ et un
score de $-0.9$ :

In [ ]:
def perte_charniere_exemple(y, score):
    return max(0.0, 1 - y * score)

for y, score in [(1, 0.4), (-1, -0.9)]:
    print(f"y={y:+d}, score={score:+.1f} -> perte charniere = {perte_charniere_exemple(y, score):.2f}")


### Exercice 3

Écrivez une fonction `hinge_loss(y, score)` **vectorisée** (avec
`np.maximum`, pour des tableaux `y` et `score` de même taille) qui calcule
$[1-y\cdot\text{score}]_+$, et vérifiez-la sur les trois couples suivants :
$(y{=}1,\,\text{score}{=}0.3)$, $(y{=}1,\,\text{score}{=}1.5)$ et
$(y{=}-1,\,\text{score}{=}0.2)$. Vérifiez ensuite que la perte charnière
**moyenne** sur `X_soft, y_soft` (recodez les étiquettes en $\{-1,+1\}$)
diminue quand `C` augmente, pour un SVM `SVC(kernel="linear", C=...)`.

In [ ]:
# Ecrivez hinge_loss(y, score) vectorisee, verifiez-la sur les 3 couples donnes,
# puis montrez que la perte charniere moyenne sur X_soft/y_soft diminue quand C augmente.


### Exercice 4

Pour $w=(2,0)$, $C=1$, $m=3$ exemples et des variables d'écart
$\xi=(0,\,0.2,\,0.8)$, calculez l'objectif marge souple
$\tfrac12\|w\|_2^2+\tfrac{C}{m}\sum_i\xi_i$ en Python (définissez `w`, `C`
et `xi`, puis assemblez les deux termes séparément avant de les
additionner).

In [ ]:
# Pour w=(2,0), C=1, xi=(0, 0.2, 0.8), calculez le terme de regularisation,
# le terme de penalite, et l'objectif total de la marge souple.


## 3. L'astuce du noyau (*kernel trick*)

La formulation duale ne fait intervenir les exemples que par leur produit
scalaire $\langle x_i,x_j\rangle$. On peut donc le remplacer par une
fonction noyau $K(x,x')$ plus générale, sans jamais calculer explicitement
une transformation $\Phi$ vers un espace de plus grande dimension
(éventuellement infinie) :

$$K(x,x') = \langle \Phi(x), \Phi(x')\rangle.$$

D'après le théorème de Mercer, toute fonction $K$ continue, symétrique
($K(x,x')=K(x',x)$) et semi-définie positive s'écrit comme un tel produit
scalaire — il suffit donc de connaître $K$, jamais $\Phi$.

Les noyaux usuels : linéaire $K(x,x')=\langle x,x'\rangle$, polynomial
$K(x,x')=(\langle x,x'\rangle+c)^p$, et gaussien (RBF)
$K(x,x')=\exp\!\big(-\|x-x'\|_2^2/(2\sigma^2)\big)$ — paramétré dans
`scikit-learn` par $\gamma=1/(2\sigma^2)$ : plus `gamma` est grand, plus on
privilégie la similarité locale entre points proches (risque de
sur-apprentissage), plus il est petit, plus la frontière est lisse (risque
de sous-apprentissage). Avec un noyau, la prédiction devient :

$$h(x') = \text{sign}\left(\sum_{i=1}^m \alpha_i y_i K(x', x_i)\right).$$

On génère des cercles concentriques (`make_circles`), clairement non
séparables linéairement, et on compare un noyau linéaire (qui échoue) à un
noyau gaussien (qui réussit).

In [ ]:
X_nl, y_nl = make_circles(n_samples=150, noise=0.08, factor=0.4, random_state=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), sharex=True, sharey=True)

clf_lin = SVC(kernel="linear", C=1)
clf_lin.fit(X_nl, y_nl)
acc_lin = accuracy_score(y_nl, clf_lin.predict(X_nl))
plot_svm_boundary(clf_lin, X_nl, y_nl, axes[0], f"Noyau linéaire (précision = {acc_lin:.2f})")

clf_rbf = SVC(kernel="rbf", C=1, gamma=2)
clf_rbf.fit(X_nl, y_nl)
acc_rbf = accuracy_score(y_nl, clf_rbf.predict(X_nl))
plot_svm_boundary(clf_rbf, X_nl, y_nl, axes[1], f"Noyau RBF (précision = {acc_rbf:.2f})")

plt.tight_layout()
plt.show()


$$ $$

**Question 5 :** Pourquoi le noyau linéaire ne peut-il pas séparer ces deux classes, quelle que soit la valeur de `C` ? La frontière obtenue avec le noyau RBF est-elle, elle aussi, un hyperplan — et si oui, dans quel espace ?

$$ $$

À `C` fixé, on fait maintenant varier `gamma` pour observer le compromis
sous-apprentissage / sur-apprentissage, sur des données en forme de lunes
entrelacées.

In [ ]:
X_moon, y_moon = make_moons(n_samples=150, noise=0.25, random_state=3)

gamma_values = [0.1, 1, 10]
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharex=True, sharey=True)

for ax, g in zip(axes, gamma_values):
    clf = SVC(kernel="rbf", C=1, gamma=g)
    clf.fit(X_moon, y_moon)
    acc = accuracy_score(y_moon, clf.predict(X_moon))
    plot_svm_boundary(clf, X_moon, y_moon, ax,
                       f"gamma = {g} (précision = {acc:.2f}, "
                       f"{len(clf.support_vectors_)} vect. support)")

plt.tight_layout()
plt.show()


$$ $$

**Question 6 :** Comment la frontière évolue-t-elle entre `gamma=0.1` et `gamma=10` ? Laquelle des trois valeurs vous semble la plus susceptible de sur-apprendre, et laquelle de sous-apprendre ?

$$ $$

$$ $$

**Question 7 :** La précision affichée est calculée sur les données d'entraînement. Pourquoi cette précision seule ne suffit-elle pas à choisir la meilleure valeur de `gamma` ?

$$ $$

### Exercice 5

Le noyau polynomial se calcule directement : pour $x=(1,2)$, $z=(3,1)$,
$c=1$ et $p=2$, $K(x,z)=(\langle x,z\rangle+c)^p=(5+1)^2=36$.

In [ ]:
def poly_kernel_manuel(x, z, c, p):
    return (np.dot(x, z) + c) ** p

k_poly = poly_kernel_manuel(np.array([1.0, 2.0]), np.array([3.0, 1.0]), c=1, p=2)
print(f"K_poly(x, z) = {k_poly:.1f}")


À vous, avec le noyau gaussien. Écrivez une fonction
`rbf_kernel_manuel(x, z, gamma)` qui calcule
$K(x,z)=\exp(-\gamma\|x-z\|_2^2)$ à la main avec `numpy` (sans utiliser
`sklearn.metrics.pairwise.rbf_kernel`), et vérifiez-la sur $x=(0,0)$,
$z=(1,1)$, $\gamma=0.5$ : vous devez retrouver $K\approx0.368$.

In [ ]:
# Ecrivez rbf_kernel_manuel(x, z, gamma) = exp(-gamma * ||x-z||_2^2), et verifiez-la
# sur x=(0,0), z=(1,1), gamma=0.5 (K attendu environ 0.368).


### Exercice 6

Comparons noyau linéaire et RBF avec un vrai découpage train/test, sur les
données en cercles concentriques :

In [ ]:
X_tr_nl, X_te_nl, y_tr_nl, y_te_nl = train_test_split(
    X_nl, y_nl, test_size=0.3, random_state=0)

for k in ["linear", "rbf"]:
    clf = SVC(kernel=k)
    clf.fit(X_tr_nl, y_tr_nl)
    acc = accuracy_score(y_te_nl, clf.predict(X_te_nl))
    print(f"noyau {k:>6} : accuracy test = {acc:.3f}")


Faites de même sur `X_moon, y_moon` (données en lunes), mais avec **trois**
noyaux : linéaire, polynomial (degré 3) et RBF, paramètres par défaut
sinon. Séparez en train/test (`test_size=0.3`, `random_state=0`), calculez
la précision de chacun sur le test, affichez les trois frontières côte à
côte avec `plot_svm_boundary`, et concluez sur le noyau le plus adapté à la
forme de ces données.

In [ ]:
# Sur X_moon/y_moon : train/test split, comparez linear / poly (degre 3) / rbf
# (accuracy test), affichez les 3 frontieres cote a cote, et concluez.


### Exercice 7

Sur `load_wine`, une recherche simple par `GridSearchCV` (noyau RBF, grille
réduite sur `C` et `gamma`) :

In [ ]:
X_w, y_w = load_wine(return_X_y=True)
X_tr_w, X_te_w, y_tr_w, y_te_w = train_test_split(
    X_w, y_w, test_size=0.3, random_state=0, stratify=y_w)

scaler_w = StandardScaler().fit(X_tr_w)
X_tr_w_sc = scaler_w.transform(X_tr_w)
X_te_w_sc = scaler_w.transform(X_te_w)

grid_w = GridSearchCV(SVC(kernel="rbf"), {"C": [0.1, 1, 10], "gamma": [0.01, 0.1, 1]}, cv=5)
grid_w.fit(X_tr_w_sc, y_tr_w)

print("Meilleurs paramètres :", grid_w.best_params_)
print(f"Accuracy test : {accuracy_score(y_te_w, grid_w.predict(X_te_w_sc)):.3f}")


À vous, sur `load_breast_cancer` : chargez les données, séparez en
train/test (`test_size=0.3`, `random_state=0`), **standardisez** avec
`StandardScaler` (fit sur le train uniquement — c'est important pour un
noyau RBF, sensible à l'échelle des variables), puis lancez un
`GridSearchCV` (`cv=5`) sur une grille plus fine de `C` (ex.
`[0.1, 1, 10, 100]`) et de `gamma` (ex. `[0.001, 0.01, 0.1, 1]`). Affichez
`best_params_` et la précision sur le test, puis commentez ce qui se passe
(sur/sous-apprentissage) si `C` ou `gamma` sont choisis très loin des
valeurs optimales trouvées.

In [ ]:
# Sur load_breast_cancer : split train/test, StandardScaler (fit sur train),
# GridSearchCV(SVC(kernel="rbf"), cv=5) sur une grille de C et gamma, affichez best_params_
# et l'accuracy test, et commentez sur/sous-apprentissage.


### Exercice 8 (bonus)

Sur le modèle `grid_w` (jeu `load_wine`) : combien de vecteurs de support
utilise-t-il, et quelle proportion cela représente-t-il par rapport au
nombre total d'exemples d'entraînement ?

In [ ]:
n_sv_w = len(grid_w.best_estimator_.support_vectors_)
print(f"Nombre de vecteurs de support (modele wine) : {n_sv_w} sur {len(X_tr_w_sc)} exemples")
print(f"Proportion : {n_sv_w / len(X_tr_w_sc):.1%}")


Faites de même sur le meilleur modèle `grid_bc.best_estimator_` trouvé à
l'exercice 7 : affichez le nombre de vecteurs de support et la proportion
qu'ils représentent. Puis vérifiez concrètement l'affirmation du cours
selon laquelle « seuls les vecteurs de support sont nécessaires à la
prédiction » : réentraînez un SVM `kernel="rbf"` avec les mêmes `C` et
`gamma`, mais uniquement sur les vecteurs de support (et leurs étiquettes)
extraits de `grid_bc.best_estimator_` (indice : `.support_` donne leurs
indices dans les données d'entraînement standardisées), et comparez sa
précision sur le test à celle du modèle original.

In [ ]:
# Sur grid_bc.best_estimator_ : nombre et proportion de vecteurs de support,
# puis reentrainez un SVM identique (memes C, gamma) sur les seuls vecteurs de support
# et comparez son accuracy test a celle du modele original.


## Pour aller plus loin

`C` et les hyperparamètres du noyau (`gamma`, `degree`...) se règlent en
pratique par validation croisée, jamais sur l'ensemble de test. Pour des
données de grande dimension (texte, images...), un noyau linéaire suffit
souvent et reste bien plus rapide qu'un noyau RBF ; `scikit-learn` propose
d'ailleurs `LinearSVC`, une implémentation dédiée au cas linéaire, qui
passe mieux à l'échelle que `SVC(kernel="linear")` sur de grands jeux de
données.